In [1]:
from pathlib import Path
import pandas as pd
import fastparquet

In [2]:
PROJECT_ROOT = Path.cwd().parent

In [10]:
dim_items = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/movies/dim_items.parquet", engine="fastparquet")
item_genres = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/movies/item_genres.parquet", engine="fastparquet")
item_keywords = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/movies/item_keywords.parquet", engine="fastparquet")
item_cast = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/movies/item_cast.parquet", engine="fastparquet")
item_directors = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/movies/item_directors.parquet", engine="fastparquet")

In [13]:
genres_agg = (
    item_genres
    .groupby("item_id")["genre"]
    .apply(lambda x: " ".join(sorted(set(x))))
    .reset_index(name="genres")
)

genres_agg.head()

,item_id,genres
0,1,Animation Comedy Family
1,2,Adventure Family Fantasy
2,3,Comedy Romance
3,4,Comedy Drama Romance
4,5,Comedy


In [14]:
keywords_agg = (
    item_keywords
    .groupby("item_id")["keyword"]
    .apply(lambda x: " ".join(sorted(set(x))))
    .reset_index(name="keywords")
)

In [15]:
directors_agg = (
    item_directors
    .groupby("item_id")["director"]
    .apply(lambda x: " ".join(sorted(set(x))))
    .reset_index(name="directors")
)

In [16]:
cast_agg = (
    item_cast
    .groupby("item_id")["actor"]
    .apply(lambda x: " ".join(sorted(set(x))))
    .reset_index(name="actors")
)

In [12]:
dim_items.head()

,item_id,source_item_id,title,description,release_date,runtime,language,popularity,vote_average,vote_count,collection_name,company_names,country_names,budget,revenue,poster_path,domain
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",1995-10-30,81.0,en,21.946943,7.7,5415.0,Toy Story Collection,Pixar Animation Studios,United States of America,30000000,373554033.0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,movie
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,1995-12-15,104.0,en,17.015539,6.9,2413.0,,TriStar Pictures Teitler Film Interscope Commu...,United States of America,65000000,262797249.0,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,movie
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,1995-12-22,101.0,en,11.712900,6.5,92.0,Grumpy Old Men Collection,Warner Bros. Lancaster Gate,United States of America,0,0.0,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,movie
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",1995-12-22,127.0,en,3.859495,6.1,34.0,,Twentieth Century Fox Film Corporation,United States of America,16000000,81452156.0,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,movie
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,1995-02-10,106.0,en,8.387519,5.7,173.0,Father of the Bride Collection,Sandollar Productions Touchstone Pictures,United States of America,0,76578911.0,/e64sOI48hQXyru7naBFyssKFxVd.jpg,movie


In [21]:
content_features.head()

,item_id,source_item_id,title,description,collection_name,company_names,country_names,domain
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Toy Story Collection,Pixar Animation Studios,United States of America,movie
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,,TriStar Pictures Teitler Film Interscope Commu...,United States of America,movie
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Grumpy Old Men Collection,Warner Bros. Lancaster Gate,United States of America,movie
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",,Twentieth Century Fox Film Corporation,United States of America,movie
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Father of the Bride Collection,Sandollar Productions Touchstone Pictures,United States of America,movie


In [20]:
content_features = dim_items[
    [
        "item_id",
        "source_item_id",
        "title",
        "description",
        "collection_name",
        "company_names",
        "country_names",
        "domain"
    ]
].copy()

In [22]:
content_features = content_features.merge(
    genres_agg,
    on="item_id",
    how="left"
)

content_features = content_features.merge(
    keywords_agg,
    on="item_id",
    how="left"
)

content_features = content_features.merge(
    directors_agg,
    on="item_id",
    how="left"
)

content_features = content_features.merge(
    cast_agg,
    on="item_id",
    how="left"
)

In [26]:
content_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 45430 entries, 0 to 45429
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   item_id          45430 non-null  int64 
 1   source_item_id   45430 non-null  int64 
 2   title            45430 non-null  string
 3   description      45430 non-null  string
 4   collection_name  45430 non-null  string
 5   company_names    45430 non-null  string
 6   country_names    45430 non-null  string
 7   domain           45430 non-null  string
 8   genres           45430 non-null  str   
 9   keywords         45430 non-null  str   
 10  directors        45430 non-null  str   
 11  actors           45430 non-null  str   
dtypes: int64(2), str(4), string(6)
memory usage: 26.5 MB


In [24]:
content_features.head()

,item_id,source_item_id,title,description,collection_name,company_names,country_names,domain,genres,keywords,directors,actors
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Toy Story Collection,Pixar Animation Studios,United States of America,movie,Animation Comedy Family,boy boy next door friends friendship jealousy ...,John Lasseter,Don Rickles Jim Varney Tim Allen Tom Hanks Wal...
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,,TriStar Pictures Teitler Film Interscope Commu...,United States of America,movie,Adventure Family Fantasy,based on children's book board game disappeara...,Joe Johnston,Bonnie Hunt Bradley Pierce Jonathan Hyde Kirst...
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Grumpy Old Men Collection,Warner Bros. Lancaster Gate,United States of America,movie,Comedy Romance,best friend duringcreditsstinger fishing old men,Howard Deutch,Ann-Margret Daryl Hannah Jack Lemmon Sophia Lo...
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",,Twentieth Century Fox Film Corporation,United States of America,movie,Comedy Drama Romance,based on novel chick flick divorce interracial...,Forest Whitaker,Angela Bassett Gregory Hines Lela Rochon Loret...
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Father of the Bride Collection,Sandollar Productions Touchstone Pictures,United States of America,movie,Comedy,aging baby confidence contraception daughter g...,Charles Shyer,Diane Keaton George Newbern Kimberly Williams-...


In [25]:
content_features = content_features.fillna("")

In [28]:
content_features["content_text"] = (
    content_features["description"].fillna("") + " " +
    content_features["genres"].fillna("") + " " +
    content_features["keywords"].fillna("") + " " +
    content_features["directors"].fillna("") + " " +
    content_features["actors"].fillna("") + " " +
    content_features["collection_name"].fillna("") + " " +
    content_features["company_names"].fillna("") + " " +
    content_features["country_names"].fillna("")
)

In [32]:
content_features["content_text"].head()

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: content_text, dtype: string

In [36]:
content_features.head()

,item_id,source_item_id,title,description,collection_name,company_names,country_names,domain,genres,keywords,directors,actors,content_text
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Toy Story Collection,Pixar Animation Studios,United States of America,movie,Animation Comedy Family,boy boy next door friends friendship jealousy ...,John Lasseter,Don Rickles Jim Varney Tim Allen Tom Hanks Wal...,"Led by Woody, Andy's toys live happily in his ..."
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,,TriStar Pictures Teitler Film Interscope Commu...,United States of America,movie,Adventure Family Fantasy,based on children's book board game disappeara...,Joe Johnston,Bonnie Hunt Bradley Pierce Jonathan Hyde Kirst...,When siblings Judy and Peter discover an encha...
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Grumpy Old Men Collection,Warner Bros. Lancaster Gate,United States of America,movie,Comedy Romance,best friend duringcreditsstinger fishing old men,Howard Deutch,Ann-Margret Daryl Hannah Jack Lemmon Sophia Lo...,A family wedding reignites the ancient feud be...
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",,Twentieth Century Fox Film Corporation,United States of America,movie,Comedy Drama Romance,based on novel chick flick divorce interracial...,Forest Whitaker,Angela Bassett Gregory Hines Lela Rochon Loret...,"Cheated on, mistreated and stepped on, the wom..."
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Father of the Bride Collection,Sandollar Productions Touchstone Pictures,United States of America,movie,Comedy,aging baby confidence contraception daughter g...,Charles Shyer,Diane Keaton George Newbern Kimberly Williams-...,Just when George Banks has recovered from his ...


In [34]:
content_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 45430 entries, 0 to 45429
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   item_id          45430 non-null  int64 
 1   source_item_id   45430 non-null  int64 
 2   title            45430 non-null  string
 3   description      45430 non-null  string
 4   collection_name  45430 non-null  string
 5   company_names    45430 non-null  string
 6   country_names    45430 non-null  string
 7   domain           45430 non-null  string
 8   genres           45430 non-null  str   
 9   keywords         45430 non-null  str   
 10  directors        45430 non-null  str   
 11  actors           45430 non-null  str   
 12  content_text     45430 non-null  string
dtypes: int64(2), str(4), string(7)
memory usage: 48.6 MB


In [ ]:
content_features.to_parquet(
    f"{PROJECT_ROOT}/data/processed/movies/content_features.parquet",
    engine="fastparquet",
    index=False
)